# LeetCode 10 — Regular Expression Matching (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def isMatch(self, s: str, p: str) -> bool:

```

**Mục tiêu sau khi học xong:**

1. Hiểu chính xác ý nghĩa của `'.'` và `'*'` trong bài, và vì sao "khớp **toàn bộ** chuỗi" quan trọng.
2. Hiểu vì sao cách "tham lam" (ăn càng nhiều càng tốt) **sai**, và vì sao phải thử nhiều lựa chọn.
3. Học **đệ quy (recursion)** từ căn bản rồi viết **Cách 1 — đệ quy thuần**.
4. Học **memoization** (ghi nhớ kết quả) và viết **Cách 2 — đệ quy + memo (top-down DP)** với `functools.lru_cache`.
5. Viết **Cách 3 — DP bảng bottom-up `dp[i][j]`**, tự tay điền bảng và trace từng ô.
6. Biết **Cách 4 — tối ưu bộ nhớ chỉ giữ 2 hàng**.
7. Nộp được bản `class Solution`, kiểm thử chéo với `re.fullmatch`, và tự giải các biến thể (thêm `'+'`, Wildcard Matching).

## 1. Đề bài

Cho chuỗi đầu vào `s` và mẫu (pattern) `p`. Hãy cài đặt phép **khớp biểu thức chính quy (regular expression matching)** hỗ trợ `'.'` và `'*'`, trong đó:

- `'.'` khớp với **một ký tự bất kỳ**.
- `'*'` khớp với **không hoặc nhiều lần** phần tử **đứng ngay trước nó**.

Phép khớp phải bao phủ **toàn bộ** chuỗi đầu vào (không phải chỉ một phần).

**Ví dụ 1**

```text
Input: s = "aa", p = "a"
Output: false
Giải thích: "a" không khớp toàn bộ chuỗi "aa".
```

**Ví dụ 2**

```text
Input: s = "aa", p = "a*"
Output: true
Giải thích: '*' nghĩa là không hoặc nhiều lần phần tử đứng trước, ở đây là 'a'.
Lặp 'a' thêm một lần, ta được "aa".
```

**Ví dụ 3**

```text
Input: s = "ab", p = ".*"
Output: true
Giải thích: ".*" nghĩa là "không hoặc nhiều lần (*) một ký tự bất kỳ (.)".
```

**Constraints (ràng buộc):**

- `1 <= s.length <= 20`
- `1 <= p.length <= 20`
- `s` chỉ chứa chữ cái tiếng Anh viết thường.
- `p` chỉ chứa chữ cái tiếng Anh viết thường, `'.'` và `'*'`.
- Đảm bảo rằng mỗi lần xuất hiện ký tự `'*'`, luôn có một ký tự hợp lệ đứng trước để nó áp dụng.

> Ràng buộc cuối rất quan trọng: pattern **không bao giờ** bắt đầu bằng `'*'` và **không có** `"**"`. Nhờ vậy mỗi `'*'` luôn "dính" vào đúng một ký tự trước nó.

### "Trọng tài" để kiểm tra: `re.fullmatch`

Python có sẵn module `re` (regular expression). Trong bài này **ta không được dùng nó để nộp** — đề bài yêu cầu ta tự cài đặt. Nhưng nó rất tiện để làm **trọng tài**: biết trước đáp án đúng để kiểm tra code của mình.

- `re.fullmatch(p, s)` trả về một object nếu `p` khớp **toàn bộ** `s`, ngược lại trả về `None`.
- `re.match(p, s)` chỉ cần khớp **phần đầu** của `s` — đó chính là kiểu "khớp một phần" mà đề **không** muốn.

Với ràng buộc của đề (chỉ có chữ thường, `.` và `*`), ý nghĩa của `.` và `*` trong `re` trùng khớp hoàn toàn với đề bài.

In [1]:
import re

examples = [("aa", "a"), ("aa", "a*"), ("ab", ".*")]
for s, p in examples:
    print(f"s={s!r:5} p={p!r:5} fullmatch -> {re.fullmatch(p, s) is not None}")

print()
print("re.match('a', 'aa')     :", re.match("a", "aa") is not None, " <- chỉ khớp phần đầu")
print("re.fullmatch('a', 'aa') :", re.fullmatch("a", "aa") is not None, "<- khớp toàn bộ, đúng ý đề")

s='aa'  p='a'   fullmatch -> False
s='aa'  p='a*'  fullmatch -> True
s='ab'  p='.*'  fullmatch -> True

re.match('a', 'aa')     : True  <- chỉ khớp phần đầu
re.fullmatch('a', 'aa') : False <- khớp toàn bộ, đúng ý đề


## 2. Hiểu đúng `'.'` và `'*'`: đọc pattern theo **token**

Điều người mới hay hiểu nhầm: `'*'` **không** đứng một mình. Nó luôn đi cặp với ký tự đứng trước.

Vì vậy ta nên đọc pattern thành từng **token** (mảnh):

| Token | Ý nghĩa | Có thể khớp |
|---|---|---|
| `a` | đúng một chữ `a` | `"a"` |
| `.` | đúng một ký tự bất kỳ | `"a"`, `"z"`, ... |
| `a*` | không hoặc nhiều chữ `a` | `""`, `"a"`, `"aa"`, `"aaa"`, ... |
| `.*` | không hoặc nhiều ký tự bất kỳ | **mọi chuỗi**, kể cả `""` |

Ví dụ pattern `"c*a*b"` gồm 3 token: `c*` · `a*` · `b`.

Lưu ý `.*` khớp `"ab"` vì mỗi lần lặp, `.` được phép khớp một ký tự **khác nhau** (`.` khớp `a`, rồi `.` khớp `b`).

Cell dưới viết một hàm nhỏ `tokenize` để tách pattern thành token, giúp ta "nhìn" pattern đúng cách.

In [2]:
def tokenize(p: str) -> list[str]:
    tokens = []
    j = 0
    while j < len(p):
        # nếu ký tự kế tiếp là '*', gộp 2 ký tự thành 1 token "x*"
        if j + 1 < len(p) and p[j + 1] == "*":
            tokens.append(p[j:j + 2])
            j += 2
        else:
            tokens.append(p[j])
            j += 1
    return tokens


for p in ["c*a*b", ".*", "a.b", "mis*is*p*."]:
    print(f"{p!r:13} -> {tokenize(p)}")

'c*a*b'       -> ['c*', 'a*', 'b']
'.*'          -> ['.*']
'a.b'         -> ['a', '.', 'b']
'mis*is*p*.'  -> ['m', 'i', 's*', 'i', 's*', 'p*', '.']


### Khớp bằng tay: `s = "aab"`, `p = "c*a*b"`

Ta cần chia `"aab"` thành các đoạn liên tiếp, mỗi token "nhận" một đoạn:

| Token | Chọn số lần | Đoạn của `s` được khớp |
|---|---|---|
| `c*` | 0 lần | `""` |
| `a*` | 2 lần | `"aa"` |
| `b` | (bắt buộc 1) | `"b"` |

Ghép lại: `"" + "aa" + "b" = "aab"` → **khớp**, kết quả `True`.

Câu hỏi then chốt của cả bài: **mỗi token `x*` nên ăn bao nhiêu ký tự?** Ta không biết trước — đó là lý do bài này khó.

## 3. Ý tưởng trực giác: "tham lam" (greedy) — và vì sao nó sai

Ý tưởng đầu tiên người mới hay nghĩ: duyệt từng token, gặp `x*` thì **ăn càng nhiều càng tốt** (greedy = tham lam), gặp token đơn thì ăn đúng 1 ký tự.

Code dưới đây cài đặt đúng ý tưởng đó (đặt tên `_wrong` để nhớ là nó sai). Ta so với trọng tài `re.fullmatch`.

In [3]:
def greedy_match_wrong(s: str, p: str) -> bool:
    i = 0                                  # vị trí đang đọc trong s
    for tok in tokenize(p):
        if len(tok) == 2:                  # token dạng "x*": ăn tối đa
            while i < len(s) and tok[0] in (s[i], "."):
                i += 1
        else:                              # token đơn: phải khớp đúng 1 ký tự
            if i < len(s) and tok in (s[i], "."):
                i += 1
            else:
                return False
    return i == len(s)                     # phải dùng hết s


for s, p in [("aab", "c*a*b"), ("aa", "a*"), ("aa", "a*a"), ("ab", ".*b")]:
    got = greedy_match_wrong(s, p)
    expected = re.fullmatch(p, s) is not None
    print(f"s={s!r:6} p={p!r:8} greedy={got!s:5} đúng={expected!s:5} {'OK' if got == expected else '<-- SAI'}")

s='aab'  p='c*a*b'  greedy=True  đúng=True  OK
s='aa'   p='a*'     greedy=True  đúng=True  OK
s='aa'   p='a*a'    greedy=False đúng=True  <-- SAI
s='ab'   p='.*b'    greedy=False đúng=True  <-- SAI


### Greedy vướng ở đâu?

Với `s = "aa"`, `p = "a*a"`:

- `a*` tham lam ăn hết cả `"aa"`.
- Token `a` cuối không còn ký tự nào để khớp → trả `False`.
- Nhưng nếu `a*` chỉ ăn **1** chữ `a`, token `a` cuối ăn chữ còn lại → khớp!

**Bài học:** với mỗi `x*` ta phải **thử nhiều khả năng** (ăn 0, 1, 2, ... ký tự) và chấp nhận nếu **có ít nhất một** khả năng dẫn tới khớp toàn bộ.

"Thử một lựa chọn, nếu không được thì quay lại thử lựa chọn khác" được viết tự nhiên nhất bằng **đệ quy**. Vì vậy trước khi giải, ta học đệ quy từ đầu.

## 4. Kiến thức nền: đệ quy (recursion)

**Đệ quy (recursion)** là khi một hàm **gọi lại chính nó** để giải một bài toán **nhỏ hơn** cùng loại.

Một hàm đệ quy luôn có 2 phần:

1. **Base case (trường hợp cơ sở):** bài toán đủ nhỏ để trả lời ngay, không cần gọi tiếp. Thiếu phần này hàm sẽ gọi mãi không dừng.
2. **Recursive case (trường hợp đệ quy):** làm một phần nhỏ công việc, rồi giao phần còn lại cho chính hàm đó.

Trước tiên ta cần 3 công cụ Python nhỏ:

- **Slicing** `s[1:]`: bỏ ký tự đầu; `s[2:]`: bỏ 2 ký tự đầu. Nếu chuỗi đã ngắn hơn thì kết quả là chuỗi rỗng `""` (không lỗi).
- **Chuỗi rỗng là "falsy"**: `not ""` là `True`, `bool("")` là `False`. Nên `if not p:` nghĩa là "nếu `p` rỗng".
- Truy cập `s[0]` khi `s` rỗng sẽ gây lỗi `IndexError` — phải kiểm tra `s` khác rỗng trước.

In [4]:
s = "aab"
print(repr(s[1:]), repr(s[2:]), repr(s[5:]))   # 'ab' 'b' ''  (cắt quá dài vẫn không lỗi)
print(not "", bool(""), bool("a"))           # True False True

try:
    ""[0]
except IndexError as e:
    print("IndexError:", e)

'ab' 'b' ''
True False True
IndexError: string index out of range


### Ví dụ mini: so sánh 2 chuỗi bằng đệ quy

Bài toán đơn giản hơn: chuỗi `s` có **bằng đúng** chuỗi `t` không? (coi như pattern không có `.` hay `*`).

Suy nghĩ đệ quy:

- **Base case:** nếu `t` rỗng → `s` cũng phải rỗng.
- **Recursive case:** ký tự đầu phải bằng nhau, **và** phần còn lại `s[1:]` phải bằng `t[1:]`.

Ta in ra mỗi lần gọi, thụt lề theo độ sâu `depth` để thấy **call stack** (ngăn xếp lời gọi): lời gọi ngoài phải **chờ** lời gọi bên trong trả kết quả về.

In [5]:
def equal_rec(s: str, t: str, depth: int = 0) -> bool:
    pad = "    " * depth
    print(f"{pad}equal_rec({s!r}, {t!r})")
    if not t:                          # base case
        return not s
    if not s or s[0] != t[0]:          # hết s hoặc ký tự đầu khác nhau
        return False
    return equal_rec(s[1:], t[1:], depth + 1)   # giao phần còn lại


print("Kết quả:", equal_rec("abc", "abc"))
print()
print("Kết quả:", equal_rec("abc", "abd"))

equal_rec('abc', 'abc')
    equal_rec('bc', 'bc')
        equal_rec('c', 'c')
            equal_rec('', '')
Kết quả: True

equal_rec('abc', 'abd')
    equal_rec('bc', 'bd')
        equal_rec('c', 'd')
Kết quả: False


### Short-circuit của `and` / `or`

Lời giải sẽ dùng các biểu thức như `A or (B and C)`. Python tính **lười (short-circuit)**:

- `X and Y`: nếu `X` là `False` thì **không tính** `Y` nữa (kết quả chắc chắn `False`).
- `X or Y`: nếu `X` là `True` thì **không tính** `Y` nữa (kết quả chắc chắn `True`).

Điều này vừa giúp **tránh lỗi** (`bool(s) and s[0] == ...` không bao giờ đụng `s[0]` khi `s` rỗng), vừa giúp **tiết kiệm**: thử nhánh 1 thành công thì khỏi thử nhánh 2.

In [6]:
def loud(name, value):
    print("  đang tính", name)
    return value

print("False and ...  ->", loud("A", False) and loud("B", True))
print("True or ...    ->", loud("C", True) or loud("D", False))
print("False or ...   ->", loud("E", False) or loud("F", True))

  đang tính A
False and ...  -> False
  đang tính C
True or ...    -> True
  đang tính E
  đang tính F
False or ...   -> True


### Bước đệm: thêm `'.'` (chưa có `'*'`)

Bây giờ pattern có thể chứa `'.'`. Ta định nghĩa:

```python
first_match = bool(s) and p[0] in (s[0], ".")
```

Đọc là: "`s` còn ký tự, **và** ký tự đầu của `p` bằng ký tự đầu của `s` **hoặc** là dấu chấm".

- `p[0] in (s[0], ".")` là cách viết gọn của `p[0] == s[0] or p[0] == "."`.
- `bool(s) and ...` bảo vệ khỏi `IndexError` nhờ short-circuit.

Biến `first_match` sẽ là "viên gạch" dùng xuyên suốt bài.

In [7]:
def match_dot_only(s: str, p: str) -> bool:
    if not p:
        return not s
    first_match = bool(s) and p[0] in (s[0], ".")
    return first_match and match_dot_only(s[1:], p[1:])


print(match_dot_only("abc", "a.c"))   # True
print(match_dot_only("abc", "..."))   # True
print(match_dot_only("abc", ".."))    # False: p hết mà s còn 'c'
print(match_dot_only("ab", "..."))    # False: s hết mà p còn '.'

True
True
False
False


## 5. Cách 1 — Đệ quy thuần (thử mọi lựa chọn cho `'*'`)

Bây giờ thêm `'*'`. Ta chỉ cần nhìn **token đầu tiên** của `p`:

**Trường hợp A — token đầu là `x*`** (tức `len(p) >= 2 and p[1] == "*"`). Có đúng 2 lựa chọn:

1. **Dùng `x*` 0 lần:** bỏ hẳn token, khớp tiếp `s` với `p[2:]`.
2. **Dùng `x*` ít nhất 1 lần:** `x` phải khớp `s[0]` (tức `first_match`), ăn `s[0]`, rồi khớp `s[1:]` với **chính `p`** (vẫn giữ `x*`).

**Trường hợp B — token đầu là ký tự đơn** (chữ hoặc `.`): `first_match` **và** khớp `s[1:]` với `p[1:]`.

### Vì sao nhánh "≥ 1 lần" lại **giữ nguyên `p`**?

"`x*` dùng ≥ 1 lần" = "ăn 1 ký tự bằng `x`" + "`x*` dùng tiếp **≥ 0** lần cho phần còn lại". Phần sau chính là `x*` một lần nữa, nên ta giữ `p`. Nhờ vậy chỉ với 2 nhánh, đệ quy tự sinh ra mọi số lần 0, 1, 2, 3, ...

Ví dụ `s = "aaa"`, `p = "a*"`: ăn `a` → `("aa", "a*")` → ăn `a` → `("a", "a*")` → ăn `a` → `("", "a*")` → dùng 0 lần → `("", "")` → `True`.

### Base case

Chỉ cần: **nếu `p` rỗng thì kết quả là `s` rỗng**. Không được dừng khi `s` rỗng, vì `s = ""` vẫn có thể khớp `p = "a*b*"`!

In [8]:
def is_match_recursive(s: str, p: str) -> bool:
    if not p:                                     # base case
        return not s

    first_match = bool(s) and p[0] in (s[0], ".")

    if len(p) >= 2 and p[1] == "*":               # token đầu là "x*"
        return (is_match_recursive(s, p[2:])      # dùng 0 lần
                or (first_match and is_match_recursive(s[1:], p)))  # dùng >= 1 lần
    return first_match and is_match_recursive(s[1:], p[1:])          # token đơn


for s, p in [("aa", "a"), ("aa", "a*"), ("ab", ".*"), ("aab", "c*a*b"), ("aa", "a*a"), ("mississippi", "mis*is*p*.")]:
    print(f"s={s!r:14} p={p!r:13} -> {is_match_recursive(s, p)}")

s='aa'           p='a'           -> False
s='aa'           p='a*'          -> True
s='ab'           p='.*'          -> True
s='aab'          p='c*a*b'       -> True
s='aa'           p='a*a'         -> True
s='mississippi'  p='mis*is*p*.'  -> False


### Trace Cách 1: cây lời gọi đệ quy

Hàm `trace_recursive` dưới đây làm y hệt `is_match_recursive` nhưng in mỗi lời gọi:

- Phần `[...]` cho biết **vì sao** có lời gọi này (bỏ token / token ăn một ký tự).
- Ký hiệu `│` thể hiện độ sâu; dòng `└─>` là **kết quả trả về** của lời gọi ở cùng độ sâu.

Ví dụ thứ hai `("aa", "a*a")` là đúng ca greedy bị sai — quan sát đệ quy **quay lui (backtrack)** để thử lựa chọn khác.

In [9]:
def trace_recursive(s: str, p: str, depth: int = 0, why: str = "gọi đầu") -> bool:
    pad = "│  " * depth
    if not p:
        res = not s
        print(f"{pad}match({s!r}, {p!r})  [{why}]  p rỗng -> {res}")
        return res
    first_match = bool(s) and p[0] in (s[0], ".")
    if len(p) >= 2 and p[1] == "*":
        print(f"{pad}match({s!r}, {p!r})  [{why}]  token '{p[:2]}', first_match={first_match}")
        res = trace_recursive(s, p[2:], depth + 1, f"bỏ '{p[:2]}' (0 lần)")
        if not res and first_match:
            res = trace_recursive(s[1:], p, depth + 1, f"'{p[:2]}' ăn '{s[0]}'")
    else:
        print(f"{pad}match({s!r}, {p!r})  [{why}]  token '{p[0]}', first_match={first_match}")
        res = first_match and trace_recursive(s[1:], p[1:], depth + 1, f"'{p[0]}' khớp '{s[0]}'")
    print(f"{pad}└─> {res}")
    return res


trace_recursive("aab", "c*a*b")
print()
trace_recursive("aa", "a*a")

match('aab', 'c*a*b')  [gọi đầu]  token 'c*', first_match=False
│  match('aab', 'a*b')  [bỏ 'c*' (0 lần)]  token 'a*', first_match=True
│  │  match('aab', 'b')  [bỏ 'a*' (0 lần)]  token 'b', first_match=False
│  │  └─> False
│  │  match('ab', 'a*b')  ['a*' ăn 'a']  token 'a*', first_match=True
│  │  │  match('ab', 'b')  [bỏ 'a*' (0 lần)]  token 'b', first_match=False
│  │  │  └─> False
│  │  │  match('b', 'a*b')  ['a*' ăn 'a']  token 'a*', first_match=False
│  │  │  │  match('b', 'b')  [bỏ 'a*' (0 lần)]  token 'b', first_match=True
│  │  │  │  │  match('', '')  ['b' khớp 'b']  p rỗng -> True
│  │  │  │  └─> True
│  │  │  └─> True
│  │  └─> True
│  └─> True
└─> True

match('aa', 'a*a')  [gọi đầu]  token 'a*', first_match=True
│  match('aa', 'a')  [bỏ 'a*' (0 lần)]  token 'a', first_match=True
│  │  match('a', '')  ['a' khớp 'a']  p rỗng -> False
│  └─> False
│  match('a', 'a*a')  ['a*' ăn 'a']  token 'a*', first_match=True
│  │  match('a', 'a')  [bỏ 'a*' (0 lần)]  token 'a', first_match

True

### Đọc trace `("aab", "c*a*b")` bằng bảng

| Bước | Lời gọi | Token đầu | `first_match` | Quyết định | Kết quả |
|---:|---|---|:---:|---|:---:|
| 1 | `('aab', 'c*a*b')` | `c*` | False | thử 0 lần; không thể ăn vì `c` ≠ `a` | True (nhờ bước 2) |
| 2 | `('aab', 'a*b')` | `a*` | True | thử 0 lần trước | |
| 3 | `('aab', 'b')` | `b` | False | `b` ≠ `a` → hỏng | False |
| 4 | `('ab', 'a*b')` | `a*` | True | quay lui: `a*` ăn `a`; thử 0 lần trước | |
| 5 | `('ab', 'b')` | `b` | False | hỏng | False |
| 6 | `('b', 'a*b')` | `a*` | False | quay lui: `a*` ăn `a`; chỉ còn cách 0 lần | |
| 7 | `('b', 'b')` | `b` | True | `b` khớp `b` | |
| 8 | `('', '')` | — | — | `p` rỗng, `s` rỗng | **True** |

Kết quả `True` được "truyền ngược" lên dần tới lời gọi đầu tiên.

### Độ phức tạp Cách 1 — vướng ở đâu?

Gọi `m = len(s)`, `n = len(p)`.

Mỗi token `x*` sinh ra 2 nhánh. Với nhiều `x*` liên tiếp, số cách chia `s` cho các token tăng **theo cấp số mũ**. Trường hợp xấu: `s = "aaa...a"` (k chữ `a`), `p = "a*a*...a*b"` (k token `a*` rồi `b`) — chắc chắn `False` nhưng phải thử **mọi** cách chia trước khi dám kết luận.

Ngoài ra mỗi lời gọi còn tạo chuỗi mới bằng slicing (tốn `O(m + n)`).

Hãy đếm số lần gọi hàm:

In [10]:
call_count = 0

def count_calls_recursive(s: str, p: str) -> bool:
    global call_count
    call_count += 1
    if not p:
        return not s
    first_match = bool(s) and p[0] in (s[0], ".")
    if len(p) >= 2 and p[1] == "*":
        return count_calls_recursive(s, p[2:]) or (first_match and count_calls_recursive(s[1:], p))
    return first_match and count_calls_recursive(s[1:], p[1:])


print(" k | kết quả | số lời gọi")
for k in range(2, 11, 2):
    call_count = 0
    res = count_calls_recursive("a" * k, "a*" * k + "b")
    print(f"{k:2} | {res!s:7} | {call_count:>9,}")

 k | kết quả | số lời gọi
 2 | False   |        15
 4 | False   |       195
 6 | False   |     2,639
 8 | False   |    37,179
10 | False   |   537,471


Tăng `k` thêm 2 thì số lời gọi nhân lên khoảng **14 lần**. Ca `s = "a" * 20`, `p = "a*" * 9 + "b"` (vẫn đúng ràng buộc: 20 và 19 ký tự) cần khoảng **24 triệu** lời gọi — quá chậm.

**Quan sát quan trọng:** trong cây lời gọi, có rất nhiều lời gọi **trùng nhau**. Ví dụ `("a", "a*b")` có thể đạt tới bằng nhiều đường khác nhau: `a*` thứ nhất ăn 1 chữ, hoặc `a*` thứ hai ăn 1 chữ, ... Mỗi lần ta lại tính lại từ đầu → lãng phí.

→ Cải tiến: **nhớ lại** kết quả đã tính. Đó là **memoization**.

## 6. Kiến thức nền: memoization (ghi nhớ kết quả)

**Memoization** = lưu kết quả của mỗi bài toán con vào một "cuốn sổ" (thường là `dict`). Lần sau gặp lại đúng bài toán con đó, chỉ cần tra sổ thay vì tính lại.

Ví dụ kinh điển: số Fibonacci `fib(n) = fib(n-1) + fib(n-2)`, `fib(0) = 0`, `fib(1) = 1`.

- Bản thường: `fib(n-1)` và `fib(n-2)` đều tự tính lại `fib(n-3)`, ... → số lời gọi bùng nổ.
- Bản memo: mỗi `n` chỉ tính **đúng một lần**.

Cell dưới so sánh số lời gọi.

In [11]:
fib_calls = 0

def fib_plain(n: int) -> int:
    global fib_calls
    fib_calls += 1
    if n <= 1:
        return n
    return fib_plain(n - 1) + fib_plain(n - 2)


memo_calls = 0
memo = {}                       # "cuốn sổ": n -> fib(n)

def fib_memo(n: int) -> int:
    global memo_calls
    memo_calls += 1
    if n in memo:               # đã tính rồi -> tra sổ
        return memo[n]
    result = n if n <= 1 else fib_memo(n - 1) + fib_memo(n - 2)
    memo[n] = result            # ghi vào sổ
    return result


print("fib_plain(25) =", fib_plain(25), "| số lời gọi:", fib_calls)
print("fib_memo(25)  =", fib_memo(25), "| số lời gọi:", memo_calls)

fib_plain(25) = 75025 | số lời gọi: 242785
fib_memo(25)  = 75025 | số lời gọi: 49


### `functools.lru_cache`: memo tự động bằng decorator

Tự quản lý `dict` hơi dài dòng. Python có sẵn `functools.lru_cache`:

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def f(...):
    ...
```

- Dòng `@lru_cache(maxsize=None)` đặt ngay trên `def` gọi là **decorator**: nó "bọc" hàm `f` lại, tự động ghi nhớ kết quả theo **tham số** truyền vào.
- `maxsize=None` nghĩa là nhớ không giới hạn.
- Tham số phải **hashable** (dùng làm key của dict được): `int`, `str`, `tuple` đều được; `list` thì không.
- `f.cache_info()` cho biết số lần trúng sổ (`hits`) và phải tính mới (`misses`).

In [12]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_cached(n: int) -> int:
    return n if n <= 1 else fib_cached(n - 1) + fib_cached(n - 2)

print(fib_cached(25))
print(fib_cached.cache_info())

75025
CacheInfo(hits=23, misses=26, maxsize=None, currsize=26)


### Áp dụng vào bài: trạng thái `(i, j)` thay cho chuỗi cắt

Trong Cách 1, mọi lời gọi đều có dạng `match(s[i:], p[j:])` — một **hậu tố (suffix)** của `s` và một hậu tố của `p`. Vậy mỗi bài toán con được xác định hoàn toàn bởi **2 số nguyên** `(i, j)`:

> `dp(i, j)` = `True` nếu `s[i:]` khớp `p[j:]`.

Chỉ có `(m + 1) × (n + 1)` cặp `(i, j)` khác nhau — rất ít (tối đa `21 × 21 = 441`)! Bảng đổi từ slicing sang chỉ số:

| Cách 1 (slicing) | Cách 2 (chỉ số) |
|---|---|
| `not p` (p rỗng) | `j == n` |
| `not s` (s rỗng) | `i == m` |
| `bool(s) and p[0] in (s[0], ".")` | `i < m and p[j] in (s[i], ".")` |
| `len(p) >= 2 and p[1] == "*"` | `j + 1 < n and p[j + 1] == "*"` |
| `match(s, p[2:])` | `dp(i, j + 2)` |
| `match(s[1:], p)` | `dp(i + 1, j)` |
| `match(s[1:], p[1:])` | `dp(i + 1, j + 1)` |

Lợi ích kép: không tạo chuỗi mới, và tham số `int` dùng làm key memo rất nhẹ.

Trước hết ta viết bản memo bằng `dict` tự quản lý (để **thấy** cuốn sổ), có đếm số lời gọi.

In [13]:
def is_match_memo_dict(s: str, p: str):
    m, n = len(s), len(p)
    memo = {}                     # (i, j) -> bool
    calls = 0

    def dp(i: int, j: int) -> bool:
        nonlocal calls
        calls += 1
        if (i, j) in memo:
            return memo[(i, j)]
        if j == n:
            ans = i == m
        else:
            first_match = i < m and p[j] in (s[i], ".")
            if j + 1 < n and p[j + 1] == "*":
                ans = dp(i, j + 2) or (first_match and dp(i + 1, j))
            else:
                ans = first_match and dp(i + 1, j + 1)
        memo[(i, j)] = ans
        return ans

    return dp(0, 0), memo, calls


result, memo, calls = is_match_memo_dict("aab", "c*a*b")
print("Kết quả:", result, "| số lời gọi:", calls)
for (i, j), v in sorted(memo.items()):
    print(f"  dp({i},{j}): s[{i}:]={'aab'[i:]!r:6} p[{j}:]={'c*a*b'[j:]!r:8} -> {v}")

Kết quả: True | số lời gọi: 8
  dp(0,0): s[0:]='aab'  p[0:]='c*a*b'  -> True
  dp(0,2): s[0:]='aab'  p[2:]='a*b'    -> True
  dp(0,4): s[0:]='aab'  p[4:]='b'      -> False
  dp(1,2): s[1:]='ab'   p[2:]='a*b'    -> True
  dp(1,4): s[1:]='ab'   p[4:]='b'      -> False
  dp(2,2): s[2:]='b'    p[2:]='a*b'    -> True
  dp(2,4): s[2:]='b'    p[4:]='b'      -> True
  dp(3,5): s[3:]=''     p[5:]=''       -> True


Mỗi dòng trong "cuốn sổ" ứng với **một** bài toán con, được tính đúng một lần. Đối chiếu với trace Cách 1: `dp(1,2)` chính là lời gọi `('ab', 'a*b')`, `dp(3,5)` là `('', '')`, ...

Bây giờ so sánh số lời gọi trên ca xấu `s = "a"*k`, `p = "a*"*k + "b"`:

In [14]:
print(" k | đệ quy thuần | đệ quy + memo")
for k in range(2, 11, 2):
    call_count = 0
    r1 = count_calls_recursive("a" * k, "a*" * k + "b")
    r2, _, memo_calls_k = is_match_memo_dict("a" * k, "a*" * k + "b")
    assert r1 == r2
    print(f"{k:2} | {call_count:>12,} | {memo_calls_k:>13,}")

 k | đệ quy thuần | đệ quy + memo
 2 |           15 |            11
 4 |          195 |            37
 6 |        2,639 |            79
 8 |       37,179 |           137


10 |      537,471 |           211


Từ hàng trăm nghìn lời gọi xuống còn vài trăm. Đây là bản **top-down DP** (quy hoạch động từ trên xuống): vẫn suy nghĩ đệ quy như Cách 1, chỉ thêm trí nhớ.

Bản gọn dùng `lru_cache` (hàm `dp` lồng bên trong để "nhìn thấy" `s`, `p`, `m`, `n`; mỗi lần gọi `is_match_memo` tạo một `dp` mới với cache riêng):

In [15]:
def is_match_memo(s: str, p: str) -> bool:
    m, n = len(s), len(p)

    @lru_cache(maxsize=None)
    def dp(i: int, j: int) -> bool:          # s[i:] có khớp p[j:] không?
        if j == n:
            return i == m
        first_match = i < m and p[j] in (s[i], ".")
        if j + 1 < n and p[j + 1] == "*":
            return dp(i, j + 2) or (first_match and dp(i + 1, j))
        return first_match and dp(i + 1, j + 1)

    return dp(0, 0)


for s, p in [("aa", "a"), ("aa", "a*"), ("ab", ".*"), ("aab", "c*a*b"), ("a" * 20, "a*" * 9 + "b")]:
    print(f"s={s!r:24} p={p!r:21} -> {is_match_memo(s, p)}")

s='aa'                     p='a'                   -> False
s='aa'                     p='a*'                  -> True
s='ab'                     p='.*'                  -> True
s='aab'                    p='c*a*b'               -> True
s='aaaaaaaaaaaaaaaaaaaa'   p='a*a*a*a*a*a*a*a*a*b' -> False


### Độ phức tạp Cách 2

Với `m = len(s)`, `n = len(p)`:

- **Thời gian:** `O(m · n)` — có `(m + 1)(n + 1)` trạng thái, mỗi trạng thái tính `O(1)` (vài phép so sánh + tra cache).
- **Bộ nhớ:** `O(m · n)` cho cache, cộng độ sâu đệ quy `O(m + n)` (mỗi lời gọi tăng `i` hoặc `j`).

Điểm yếu còn lại: vẫn dùng đệ quy (với input lớn có thể chạm giới hạn độ sâu của Python, ~1000), và khó "nhìn" toàn cảnh. → Ta lật ngược lại: tự điền bảng từ bài toán nhỏ nhất.

## 7. Cách 3 — DP bảng bottom-up `dp[i][j]`

**Bottom-up DP** (quy hoạch động từ dưới lên): thay vì gọi đệ quy rồi nhớ, ta tạo sẵn một **bảng 2 chiều** và điền lần lượt từ ô nhỏ đến ô lớn, sao cho khi điền một ô thì các ô nó cần **đã có sẵn**.

Để bảng đọc tự nhiên từ trái sang phải, ta đổi góc nhìn từ hậu tố sang **tiền tố (prefix)**:

> `dp[i][j]` = `True` nếu `s[:i]` (i ký tự đầu của `s`) khớp `p[:j]` (j ký tự đầu của `p`).

- Bảng có `m + 1` hàng (`i = 0..m`) và `n + 1` cột (`j = 0..n`). Hàng/cột 0 ứng với chuỗi rỗng.
- **Đáp án** nằm ở ô góc dưới phải: `dp[m][n]`.
- Cẩn thận lệch chỉ số: ô `dp[i][j]` nói về ký tự **cuối** của tiền tố là `s[i-1]` và `p[j-1]`.

Tạo bảng 2 chiều trong Python: `[[False] * (n + 1) for _ in range(m + 1)]` (mỗi hàng là một list **riêng**; đừng viết `[[False] * (n + 1)] * (m + 1)` — các hàng sẽ là **cùng một** list).

In [16]:
good = [[False] * 3 for _ in range(2)]
bad = [[False] * 3] * 2

good[0][0] = True
bad[0][0] = True
print("good:", good)   # chỉ hàng 0 đổi
print("bad :", bad)    # cả 2 hàng đổi vì là cùng một list!

good: [[True, False, False], [False, False, False]]
bad : [[True, False, False], [True, False, False]]


### Công thức chuyển (transition)

Ta nhìn **ký tự cuối** của `p[:j]`, tức `p[j-1]`, và ký tự cuối của `s[:i]`, tức `s[i-1]`:

**Trường hợp 1 — `p[j-1]` là chữ hoặc `'.'`:** ký tự cuối hai bên phải khớp, và phần trước phải khớp:

```python
dp[i][j] = p[j-1] in (s[i-1], ".") and dp[i-1][j-1]
```

**Trường hợp 2 — `p[j-1] == '*'`:** token cuối là `x*` với `x = p[j-2]`. Hai lựa chọn (giống hệt Cách 1, chỉ nhìn từ cuối):

- **0 lần:** bỏ token `x*` (2 ký tự) → `dp[i][j-2]`
- **≥ 1 lần:** `x` khớp `s[i-1]`, ăn nó, **giữ** `x*` → `x in (s[i-1], ".") and dp[i-1][j]`

```python
dp[i][j] = dp[i][j-2] or (p[j-2] in (s[i-1], ".") and dp[i-1][j])
```

Ô `dp[i][j]` chỉ cần các ô ở **hàng trên** (`i-1`) hoặc **bên trái** cùng hàng (`j-2`) → điền theo thứ tự hàng từ trên xuống, mỗi hàng từ trái sang phải là đủ.

### Base case (hàng 0 và cột 0)

- `dp[0][0] = True`: chuỗi rỗng khớp pattern rỗng.
- `dp[i][0] = False` với `i ≥ 1`: pattern rỗng không khớp chuỗi khác rỗng.
- **Hàng 0** (`s` rỗng): `dp[0][j] = True` chỉ khi `p[:j]` toàn token `x*` dùng 0 lần, tức `p[j-1] == '*' and dp[0][j-2]`.

Ví dụ số: `p = "c*a*b"` → hàng 0 là `T . T . T .` (rỗng khớp `""`, `"c*"`, `"c*a*"`; không khớp `"c"`, `"c*a"`, `"c*a*b"`).

In [17]:
def dp_row0(p: str) -> list[bool]:
    n = len(p)
    row = [False] * (n + 1)
    row[0] = True
    for j in range(2, n + 1):          # j=1 không thể là '*' (ràng buộc đề)
        if p[j - 1] == "*":
            row[j] = row[j - 2]
    return row


for p in ["c*a*b", "a*b*c*", "ab*"]:
    print(f"{p!r:9}", " ".join("T" if v else "." for v in dp_row0(p)))

'c*a*b'   T . T . T .
'a*b*c*'  T . T . T . T
'ab*'     T . . .


In [18]:
def is_match_dp(s: str, p: str) -> bool:
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for j in range(2, n + 1):                         # hàng 0: s rỗng
        if p[j - 1] == "*":
            dp[0][j] = dp[0][j - 2]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == "*":
                zero = dp[i][j - 2]                                   # x* dùng 0 lần
                more = p[j - 2] in (s[i - 1], ".") and dp[i - 1][j]   # x* ăn s[i-1]
                dp[i][j] = zero or more
            else:
                dp[i][j] = p[j - 1] in (s[i - 1], ".") and dp[i - 1][j - 1]
    return dp[m][n]


for s, p in [("aa", "a"), ("aa", "a*"), ("ab", ".*"), ("aab", "c*a*b"), ("mississippi", "mis*is*p*.")]:
    print(f"s={s!r:14} p={p!r:13} -> {is_match_dp(s, p)}")

s='aa'           p='a'           -> False
s='aa'           p='a*'          -> True
s='ab'           p='.*'          -> True
s='aab'          p='c*a*b'       -> True
s='mississippi'  p='mis*is*p*.'  -> False


### Nhìn bảng DP

Hàm `build_dp_table` trả về cả bảng (cùng logic `is_match_dp`), `print_dp_table` in bảng với `T` = True, `.` = False. Hàng là các ký tự của `s`, cột là các ký tự của `p` (hàng/cột `''` là tiền tố rỗng).

In [19]:
def build_dp_table(s: str, p: str) -> list[list[bool]]:
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for j in range(2, n + 1):
        if p[j - 1] == "*":
            dp[0][j] = dp[0][j - 2]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == "*":
                dp[i][j] = dp[i][j - 2] or (p[j - 2] in (s[i - 1], ".") and dp[i - 1][j])
            else:
                dp[i][j] = p[j - 1] in (s[i - 1], ".") and dp[i - 1][j - 1]
    return dp


def print_dp_table(s: str, p: str) -> None:
    dp = build_dp_table(s, p)
    print("      ''" + "".join(f"{c:>4}" for c in p))
    for i in range(len(s) + 1):
        label = "''" if i == 0 else s[i - 1]
        print(f"{label:>4}" + "".join(f"{'T' if v else '.':>4}" for v in dp[i]))


print_dp_table("aab", "c*a*b")

      ''   c   *   a   *   b
  ''   T   .   T   .   T   .
   a   .   .   .   T   T   .
   a   .   .   .   .   T   .
   b   .   .   .   .   .   T


### Đọc bảng `s = "aab"`, `p = "c*a*b"`

| `i` \ `j` | 0 `''` | 1 `c` | 2 `*` | 3 `a` | 4 `*` | 5 `b` |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| 0 `''` | T | . | T | . | T | . |
| 1 `a` | . | . | . | T | T | . |
| 2 `a` | . | . | . | . | T | . |
| 3 `b` | . | . | . | . | . | **T** |

Giải thích vài ô tiêu biểu:

- `dp[1][3]` (`"a"` vs `"c*a"`): `p[2] = 'a'` khớp `s[0] = 'a'` và `dp[0][2] = T` (rỗng khớp `"c*"`) → **T**.
- `dp[2][4]` (`"aa"` vs `"c*a*"`): 0 lần → `dp[2][2] = .`; ≥1 lần → `'a'` khớp `s[1] = 'a'` và `dp[1][4] = T` → **T**.
- `dp[3][4]` (`"aab"` vs `"c*a*"`): 0 lần → `dp[3][2] = .`; ≥1 lần → `'a'` ≠ `'b'` → **.**
- `dp[3][5]` (`"aab"` vs `"c*a*b"`): `'b'` khớp `'b'` và `dp[2][4] = T` → **T**. Đáp án `True`.

Hàm trace dưới đây in quyết định cho **từng ô** (T/F) để bạn tự đối chiếu với bảng.

In [20]:
def trace_dp(s: str, p: str) -> bool:
    tf = lambda v: "T" if v else "F"
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for j in range(2, n + 1):
        if p[j - 1] == "*":
            dp[0][j] = dp[0][j - 2]
    print("Hàng 0:", " ".join(tf(v) for v in dp[0]))
    for i in range(1, m + 1):
        c = s[i - 1]
        for j in range(1, n + 1):
            if p[j - 1] == "*":
                x = p[j - 2]
                zero = dp[i][j - 2]
                ok = x in (c, ".")
                more = ok and dp[i - 1][j]
                dp[i][j] = zero or more
                print(f"dp[{i}][{j}] '{c}' vs '{x}*': 0 lần dp[{i}][{j-2}]={tf(zero)}; "
                      f">=1 lần: khớp={tf(ok)}, dp[{i-1}][{j}]={tf(dp[i-1][j])} -> {tf(dp[i][j])}")
            else:
                ok = p[j - 1] in (c, ".")
                dp[i][j] = ok and dp[i - 1][j - 1]
                print(f"dp[{i}][{j}] '{c}' vs '{p[j-1]}' : khớp={tf(ok)}, dp[{i-1}][{j-1}]={tf(dp[i-1][j-1])} -> {tf(dp[i][j])}")
    return dp[m][n]


print("Kết quả:", trace_dp("aab", "c*a*b"))

Hàng 0: T F T F T F
dp[1][1] 'a' vs 'c' : khớp=F, dp[0][0]=T -> F
dp[1][2] 'a' vs 'c*': 0 lần dp[1][0]=F; >=1 lần: khớp=F, dp[0][2]=T -> F
dp[1][3] 'a' vs 'a' : khớp=T, dp[0][2]=T -> T
dp[1][4] 'a' vs 'a*': 0 lần dp[1][2]=F; >=1 lần: khớp=T, dp[0][4]=T -> T
dp[1][5] 'a' vs 'b' : khớp=F, dp[0][4]=T -> F
dp[2][1] 'a' vs 'c' : khớp=F, dp[1][0]=F -> F
dp[2][2] 'a' vs 'c*': 0 lần dp[2][0]=F; >=1 lần: khớp=F, dp[1][2]=F -> F
dp[2][3] 'a' vs 'a' : khớp=T, dp[1][2]=F -> F
dp[2][4] 'a' vs 'a*': 0 lần dp[2][2]=F; >=1 lần: khớp=T, dp[1][4]=T -> T
dp[2][5] 'a' vs 'b' : khớp=F, dp[1][4]=T -> F
dp[3][1] 'b' vs 'c' : khớp=F, dp[2][0]=F -> F
dp[3][2] 'b' vs 'c*': 0 lần dp[3][0]=F; >=1 lần: khớp=F, dp[2][2]=F -> F
dp[3][3] 'b' vs 'a' : khớp=F, dp[2][2]=F -> F
dp[3][4] 'b' vs 'a*': 0 lần dp[3][2]=F; >=1 lần: khớp=F, dp[2][4]=T -> F
dp[3][5] 'b' vs 'b' : khớp=T, dp[2][4]=T -> T
Kết quả: True


### Độ phức tạp Cách 3

- **Thời gian:** `O(m · n)` — hai vòng lặp lồng nhau, mỗi ô `O(1)`.
- **Bộ nhớ:** `O(m · n)` cho bảng.

So với Cách 2: cùng độ phức tạp nhưng **không đệ quy** (không lo giới hạn độ sâu), luôn điền đủ mọi ô (Cách 2 có thể bỏ qua các ô không cần tới).

## 8. Cách 4 — Tối ưu bộ nhớ: chỉ giữ 2 hàng

Nhìn lại công thức: ô ở hàng `i` chỉ cần

- hàng **trên** `i - 1` (`dp[i-1][j-1]`, `dp[i-1][j]`), và
- chính hàng `i` ở cột bên trái (`dp[i][j-2]`).

Vậy không cần giữ cả bảng: chỉ cần `prev` (hàng trên) và `cur` (hàng đang điền). Xong một hàng thì `prev = cur`.

Lưu ý:

- `cur` phải là list **mới** mỗi hàng, với `cur[0] = False` (pattern rỗng không khớp `s[:i]` khác rỗng).
- Bộ nhớ giảm từ `O(m · n)` xuống `O(n)`.

In [21]:
def is_match_two_rows(s: str, p: str) -> bool:
    m, n = len(s), len(p)
    prev = [False] * (n + 1)                      # hàng 0
    prev[0] = True
    for j in range(2, n + 1):
        if p[j - 1] == "*":
            prev[j] = prev[j - 2]

    for i in range(1, m + 1):
        cur = [False] * (n + 1)                   # cur[0] = False
        for j in range(1, n + 1):
            if p[j - 1] == "*":
                cur[j] = cur[j - 2] or (p[j - 2] in (s[i - 1], ".") and prev[j])
            else:
                cur[j] = p[j - 1] in (s[i - 1], ".") and prev[j - 1]
        prev = cur
    return prev[n]


for s, p in [("aa", "a"), ("aa", "a*"), ("ab", ".*"), ("aab", "c*a*b"), ("mississippi", "mis*is*p*.")]:
    print(f"s={s!r:14} p={p!r:13} -> {is_match_two_rows(s, p)}")

s='aa'           p='a'           -> False
s='aa'           p='a*'          -> True
s='ab'           p='.*'          -> True
s='aab'          p='c*a*b'       -> True
s='mississippi'  p='mis*is*p*.'  -> False


Với `n ≤ 20`, tiết kiệm bộ nhớ ở bài này không đáng kể — Cách 4 chủ yếu để **học kỹ thuật** "chỉ giữ hàng trước" dùng lại cho rất nhiều bài DP 2 chiều khác (Edit Distance, LCS, ...). Khi phỏng vấn, Cách 3 rõ ràng và dễ giải thích hơn.

## 9. Code nộp LeetCode

Ta chọn **Cách 3 (DP bảng bottom-up)** làm bản nộp chính: không đệ quy, rõ ràng, `O(m · n)`.

Chuyển từ hàm demo sang LeetCode chỉ cần đặt logic vào phương thức `isMatch` của `class Solution` (tham số đầu là `self`), giữ đúng signature của đề.

In [22]:
class Solution:
    def isMatch(self, s: str, p: str) -> bool:
        m, n = len(s), len(p)
        dp = [[False] * (n + 1) for _ in range(m + 1)]
        dp[0][0] = True
        for j in range(2, n + 1):
            if p[j - 1] == "*":
                dp[0][j] = dp[0][j - 2]

        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if p[j - 1] == "*":
                    dp[i][j] = dp[i][j - 2] or (
                        p[j - 2] in (s[i - 1], ".") and dp[i - 1][j]
                    )
                else:
                    dp[i][j] = p[j - 1] in (s[i - 1], ".") and dp[i - 1][j - 1]
        return dp[m][n]

### Bộ test bằng `assert`

Các nhóm test:

1. Ba ví dụ của đề.
2. `'*'` dùng 0 lần ở đầu / giữa / cuối: `"c*a*b"`, `"ab*"`.
3. Ca bẫy greedy: `"a*a"`, `".*b"`, `"ab*a*c*a"`.
4. `'.'` kết hợp `'*'`: `".*c"`, `".*..a*"`, `".*a*a"`.
5. Độ dài lệch: pattern dài hơn chuỗi, chuỗi dài hơn pattern.
6. Biên: 1 ký tự; độ dài tối đa 20.

In [23]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}")


sol = Solution()
tests = [
    ("đề VD1", "aa", "a", False),
    ("đề VD2", "aa", "a*", True),
    ("đề VD3", "ab", ".*", True),
    ("0 lần ở đầu", "aab", "c*a*b", True),
    ("0 lần ở cuối", "a", "ab*", True),
    ("bẫy greedy a*a", "aaa", "a*a", True),
    ("bẫy greedy .*b", "ab", ".*b", True),
    ("nhiều *", "aaa", "ab*a*c*a", True),
    (".* rồi thiếu c", "ab", ".*c", False),
    ("cần >= 2 ký tự", "a", ".*..a*", False),
    (".*a*a", "bbbba", ".*a*a", True),
    ("mississippi F", "mississippi", "mis*is*p*.", False),
    ("mississippi T", "mississippi", "mis*is*ip*.", True),
    ("pattern dài hơn", "aaa", "aaaa", False),
    ("1 ký tự '.'", "a", ".", True),
    ("không khớp đầu", "abcd", "d*", False),
    ("dài 20, khớp", "a" * 20, "a*" * 10, True),
    ("dài 20, không", "a" * 20, "a*" * 9 + "b", False),
]
for name, s, p, expected in tests:
    check(name, sol.isMatch(s, p), expected)

PASS  đề VD1
PASS  đề VD2
PASS  đề VD3
PASS  0 lần ở đầu
PASS  0 lần ở cuối
PASS  bẫy greedy a*a
PASS  bẫy greedy .*b
PASS  nhiều *
PASS  .* rồi thiếu c
PASS  cần >= 2 ký tự
PASS  .*a*a
PASS  mississippi F
PASS  mississippi T
PASS  pattern dài hơn
PASS  1 ký tự '.'
PASS  không khớp đầu
PASS  dài 20, khớp
PASS  dài 20, không


## 10. Test đối chiếu chéo trên input ngẫu nhiên

Ta sinh ngẫu nhiên nhiều cặp `(s, p)` nhỏ **đúng ràng buộc đề** (pattern ghép từ các token `a`, `b`, `.`, mỗi token có thể kèm `*`, nên không bao giờ có `'*'` đứng đầu hay `"**"`), rồi so **cả 4 cách + `Solution`** với trọng tài `re.fullmatch`.

`random.seed(10)` cố định để lần chạy nào cũng ra cùng bộ test.

In [24]:
import random

random.seed(10)

def random_pattern(max_tokens: int = 5) -> str:
    parts = []
    for _ in range(random.randint(1, max_tokens)):
        ch = random.choice("ab.")
        parts.append(ch + ("*" if random.random() < 0.5 else ""))
    return "".join(parts)


solvers = {
    "recursive": is_match_recursive,
    "memo": is_match_memo,
    "dp": is_match_dp,
    "two_rows": is_match_two_rows,
    "Solution": Solution().isMatch,
}
n_tests, n_true = 3000, 0
for _ in range(n_tests):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 7)))
    p = random_pattern()
    expected = re.fullmatch(p, s) is not None
    n_true += expected
    for name, f in solvers.items():
        assert f(s, p) == expected, (name, s, p, expected)

print(f"{n_tests} test ngẫu nhiên PASS cho cả {len(solvers)} lời giải ({n_true} ca True, {n_tests - n_true} ca False)")

3000 test ngẫu nhiên PASS cho cả 5 lời giải (1065 ca True, 1935 ca False)


## 11. Vì sao thuật toán đúng?

**Invariant (bất biến) của bảng:** sau khi điền, `dp[i][j]` đúng bằng câu trả lời cho "`s[:i]` có khớp `p[:j]` không?".

Ta kiểm tra rằng công thức **không bỏ sót** và **không nhận nhầm**:

1. **Token cuối là ký tự đơn (`chữ` hoặc `.`):** mọi cách khớp `s[:i]` với `p[:j]` đều bắt buộc token cuối "ăn" đúng ký tự cuối `s[i-1]`, phần còn lại `s[:i-1]` phải khớp `p[:j-1]`. Đây chính là công thức — không có khả năng nào khác.
2. **Token cuối là `x*`:** trong một cách khớp bất kỳ, `x*` ăn **k** ký tự cuối của `s[:i]`.
   - Nếu `k = 0` → `s[:i]` khớp `p[:j-2]` → được bắt bởi `dp[i][j-2]`.
   - Nếu `k ≥ 1` → `s[i-1]` khớp `x`, và `s[:i-1]` khớp `p[:j]` với `x*` ăn `k - 1` ký tự (vẫn là "`x*` ăn ≥ 0") → được bắt bởi `dp[i-1][j]`.

   Hai nhóm `k = 0` và `k ≥ 1` **phủ hết** mọi khả năng, nên lấy `or` là đúng.
3. **Thứ tự điền:** mỗi ô chỉ đọc ô ở hàng trên hoặc bên trái cùng hàng — đều đã điền xong → không đọc giá trị "chưa tính".
4. **Base case** đúng như đã chứng minh ở mục 7, và quy nạp theo `(i, j)` cho thấy mọi ô đều đúng. Đáp án là `dp[m][n]`.

Cách 1 và Cách 2 dựa trên đúng lập luận này (nhìn token **đầu** thay vì token cuối). Cách 4 tính ra **cùng giá trị** với Cách 3, chỉ không giữ các hàng cũ.

## 12. So sánh các cách

Với `m = len(s)`, `n = len(p)`:

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| Greedy (sai) | `x*` ăn tối đa | `O(m + n)` | `O(n)` | **Sai** với `"a*a"`, `".*b"` |
| 1. Đệ quy thuần | thử 0 lần / ≥1 lần | mũ (worst case) | `O((m + n)²)` do slicing trên stack | dễ nghĩ nhất, là nền cho mọi cách sau |
| 2. Đệ quy + memo | nhớ `dp(i, j)` | `O(m · n)` | `O(m · n)` + stack `O(m + n)` | viết nhanh từ Cách 1; chỉ tính ô cần thiết |
| 3. DP bảng bottom-up | điền `dp[i][j]` tiền tố | `O(m · n)` | `O(m · n)` | **bản nộp chính**, không đệ quy, dễ trace |
| 4. DP 2 hàng | chỉ giữ `prev`, `cur` | `O(m · n)` | `O(n)` | tối ưu bộ nhớ, kỹ thuật dùng lại nhiều |

**Lộ trình tư duy:** greedy sai → phải thử mọi lựa chọn (đệ quy) → lựa chọn trùng lặp (memo) → lật ngược thành bảng (bottom-up) → bỏ hàng cũ (2 hàng).

## 13. Các lỗi người mới hay gặp

### Lỗi 1: Base case dừng khi `s` rỗng

```python
if not s:            # SAI
    return not p
```

**Vì sao sai:** `s` rỗng vẫn có thể khớp pattern như `"a*"`, `"a*b*"` (mọi token `x*` dùng 0 lần). **Sửa:** base case chỉ dựa vào `p` rỗng: `if not p: return not s`, và bảo vệ `s[0]` bằng `bool(s) and ...`.

### Lỗi 2: Nhánh "≥ 1 lần" bỏ luôn token `x*`

```python
return match(s, p[2:]) or (first_match and match(s[1:], p[2:]))   # SAI
```

**Vì sao sai:** như vậy `x*` chỉ dùng được **0 hoặc 1** lần. `"aa"` với `"a*"` sẽ ra `False`. **Sửa:** nhánh ≥ 1 lần phải **giữ** `p`: `match(s[1:], p)`.

### Lỗi 3: Quên khởi tạo hàng 0 của bảng DP

Nếu chỉ đặt `dp[0][0] = True` mà bỏ vòng `dp[0][j] = dp[0][j-2]`, thì các pattern mở đầu bằng `x*` (như `"c*a*b"`, `"a*b"`) bị coi là không thể khớp phần đầu rỗng → `"b"` với `"a*b"` ra `False`.

### Lỗi 4: Kiểm tra `'*'` ở sai vị trí

Trong đệ quy, phải **nhìn trước** `p[1] == '*'` (hoặc `p[j+1]`) **trước khi** khớp `p[0]` như ký tự đơn. Nếu khớp `p[0]` trước rồi mới thấy `'*'`, ta đã ép token `x*` dùng ít nhất 1 lần.

### Lỗi 5: Tạo bảng bằng `[[False] * (n + 1)] * (m + 1)`

Các hàng dùng chung một list (đã demo ở mục 7) → điền một hàng là "điền" luôn mọi hàng.

### Lỗi 6: Nhầm với Wildcard Matching (#44)

Ở bài này `'*'` **gắn với ký tự trước**. Ở Wildcard Matching, `'*'` đứng một mình và khớp **mọi chuỗi**. Đọc kỹ đề trước khi viết công thức.

Cell dưới chạy 3 phiên bản sai (Lỗi 1, 2, 3) để thấy chúng thất bại ở đâu.

In [25]:
def wrong_base_case(s: str, p: str) -> bool:
    if not s:                                   # SAI: dừng khi s rỗng
        return not p
    if not p:
        return False
    first_match = p[0] in (s[0], ".")
    if len(p) >= 2 and p[1] == "*":
        return wrong_base_case(s, p[2:]) or (first_match and wrong_base_case(s[1:], p))
    return first_match and wrong_base_case(s[1:], p[1:])


def wrong_star_drop(s: str, p: str) -> bool:
    if not p:
        return not s
    first_match = bool(s) and p[0] in (s[0], ".")
    if len(p) >= 2 and p[1] == "*":             # SAI: nhánh >= 1 lần bỏ luôn x*
        return wrong_star_drop(s, p[2:]) or (first_match and wrong_star_drop(s[1:], p[2:]))
    return first_match and wrong_star_drop(s[1:], p[1:])


def wrong_no_row0(s: str, p: str) -> bool:
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True                             # SAI: thiếu vòng khởi tạo hàng 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == "*":
                dp[i][j] = dp[i][j - 2] or (p[j - 2] in (s[i - 1], ".") and dp[i - 1][j])
            else:
                dp[i][j] = p[j - 1] in (s[i - 1], ".") and dp[i - 1][j - 1]
    return dp[m][n]


for s, p in [("aa", "a*"), ("b", "a*b"), ("aab", "c*a*b")]:
    right = is_match_dp(s, p)
    print(f"s={s!r:6} p={p!r:8} đúng={right!s:5} | base_case={wrong_base_case(s, p)!s:5} "
          f"star_drop={wrong_star_drop(s, p)!s:5} no_row0={wrong_no_row0(s, p)!s:5}")

s='aa'   p='a*'     đúng=True  | base_case=False star_drop=False no_row0=False
s='b'    p='a*b'    đúng=True  | base_case=True  star_drop=True  no_row0=False
s='aab'  p='c*a*b'  đúng=True  | base_case=True  star_drop=False no_row0=False


## 14. Mẹo nhận dạng pattern

Khi gặp bài có các dấu hiệu sau, hãy nghĩ ngay tới **DP 2 chiều trên hai chuỗi `dp[i][j]`**:

- Có **hai chuỗi** (hoặc chuỗi + pattern) và câu hỏi về "khớp / biến đổi / chung" giữa chúng.
- Mỗi bước, ta chỉ xử lý **phần đầu** (hoặc **phần cuối**) của mỗi chuỗi rồi thu nhỏ bài toán.
- Có **lựa chọn** (ví dụ "dùng 0 lần hay thêm 1 lần") → đệ quy nhiều nhánh → dễ bùng nổ → **memo**.

Công thức 3 bước rất hay dùng:

1. Viết đệ quy trên `(s[i:], p[j:])` với base case rõ ràng.
2. Thêm `@lru_cache` trên `dp(i, j)` → `O(m · n)`.
3. (Nếu cần) lật thành bảng bottom-up, rồi tối ưu 2 hàng.

Riêng với bài này: **mỗi `x*` chỉ cần 2 nhánh** (0 lần / ăn 1 ký tự và giữ nguyên `x*`), không cần vòng lặp thử k = 0, 1, 2, ...

## 15. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 44 — Wildcard Matching | `'?'` khớp 1 ký tự, `'*'` khớp chuỗi bất kỳ (đứng một mình). Cùng khung `dp[i][j]`, công thức `'*'` khác. |
| LeetCode 72 — Edit Distance | `dp[i][j]` trên hai tiền tố, 3 lựa chọn: thêm / xoá / thay. |
| LeetCode 1143 — Longest Common Subsequence | `dp[i][j]` trên hai tiền tố, lấy max. |
| LeetCode 115 — Distinct Subsequences | `dp[i][j]` đếm số cách thay vì True/False. |
| LeetCode 97 — Interleaving String | `dp[i][j]` True/False trên hai chuỗi + chuỗi thứ ba. |

Biến thể tự chế để luyện (có trong bài tập): thêm toán tử `'+'` = "**một** hoặc nhiều lần phần tử đứng trước".

## 16. Bài tập vận dụng

Làm theo thứ tự từ dễ đến khó. Gợi ý ở mục 17, đáp án ở mục 18 (cuối notebook) — hãy tự làm trước!

### Bài 1 — Đoán output (không chạy code)

Mỗi cặp sau trả về `True` hay `False`?

1. `s = "aa"`, `p = "a"`
2. `s = "aa"`, `p = ".*"`
3. `s = "ab"`, `p = ".*c"`
4. `s = "aab"`, `p = "c*a*b"`
5. `s = "a"`, `p = ".*..a*"`
6. `s = "aaa"`, `p = "ab*a*c*a"`

### Bài 2 — Tính tay bảng DP

Điền bảng `dp[i][j]` (Cách 3) cho `s = "aa"`, `p = "a*a"` (bảng 3 hàng × 4 cột). Ô nào là đáp án? Giá trị là gì? Với mỗi ô ở cột `*`, ghi rõ nhánh nào (0 lần / ≥ 1 lần) cho ra `T`.

### Bài 3 — Điền TODO: đệ quy + memo

Hoàn thiện hàm theo Cách 2. Chỉ dùng chỉ số `i`, `j`, không dùng slicing.

In [26]:
from functools import lru_cache

def exercise_is_match_memo(s: str, p: str) -> bool:
    m, n = len(s), len(p)

    @lru_cache(maxsize=None)
    def dp(i: int, j: int) -> bool:
        # TODO 1: base case khi pattern đã hết (j == n)
        # TODO 2: tính first_match (nhớ kiểm tra i < m)
        # TODO 3: nếu p[j + 1] == '*': thử 0 lần hoặc >= 1 lần
        # TODO 4: ngược lại: token đơn
        pass

    return dp(0, 0)

# Tự làm xong rồi mới mở test:
# assert exercise_is_match_memo("aa", "a") == False
# assert exercise_is_match_memo("aa", "a*") == True
# assert exercise_is_match_memo("aab", "c*a*b") == True
# assert exercise_is_match_memo("mississippi", "mis*is*p*.") == False

### Bài 4 — Tự viết lại DP bottom-up

Không nhìn lại mục 7, tự viết `exercise_is_match_dp(s, p)` theo Cách 3: tạo bảng, khởi tạo hàng 0, điền theo công thức, trả `dp[m][n]`.

In [27]:
def exercise_is_match_dp(s: str, p: str) -> bool:
    # TODO: tạo bảng (m + 1) x (n + 1), khởi tạo dp[0][0] và hàng 0
    # TODO: điền từng ô theo 2 trường hợp p[j - 1] == '*' hoặc không
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_is_match_dp("b", "a*b") == True
# assert exercise_is_match_dp("ab", ".*c") == False
# assert exercise_is_match_dp("aaa", "ab*a*c*a") == True

### Bài 5 — Biến thể: thêm toán tử `'+'`

Mở rộng bài toán: pattern có thể chứa `'+'` nghĩa là "**một hoặc nhiều lần** phần tử đứng trước" (giống `re`). Mỗi `'+'` và `'*'` vẫn luôn có một ký tự hợp lệ (chữ hoặc `.`) đứng trước.

Ví dụ: `"aa"` khớp `"a+"`; `s = "b"` **không** khớp `"a+b"` (vì `a+` cần ít nhất một chữ `a`); `"ab"` khớp `".+"`.

Gợi ý: viết theo kiểu Cách 2 (memo). Kiểm tra bằng `re.fullmatch`.

In [28]:
def exercise_is_match_plus(s: str, p: str) -> bool:
    # TODO: như Cách 2, thêm một nhánh cho token "x+"
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_is_match_plus("aa", "a+") == True
# assert exercise_is_match_plus("b", "a+b") == False
# assert exercise_is_match_plus("ab", ".+") == True
# assert exercise_is_match_plus("aab", "a+a*b+") == True

### Bài 6 — Mini challenge: Wildcard Matching (LeetCode 44)

Pattern gồm chữ thường, `'?'` và `'*'`, trong đó:

- `'?'` khớp **một ký tự bất kỳ**.
- `'*'` khớp **chuỗi bất kỳ** (kể cả rỗng) — **đứng một mình**, không gắn với ký tự trước.

Viết `exercise_wildcard(s, p)` bằng DP bảng `dp[i][j]` (tiền tố). Kiểm tra bằng `fnmatch.fnmatchcase(s, p)` của thư viện chuẩn (với pattern chỉ có chữ, `?`, `*` thì ý nghĩa giống hệt).

In [29]:
def exercise_wildcard(s: str, p: str) -> bool:
    # TODO 1: dp[0][0] = True; hàng 0: chỉ toàn '*' mới khớp chuỗi rỗng
    # TODO 2: p[j - 1] == '*': '*' khớp rỗng HOẶC '*' ăn thêm s[i - 1]
    # TODO 3: p[j - 1] là chữ hoặc '?': khớp ký tự cuối và dp[i - 1][j - 1]
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_wildcard("aa", "a") == False
# assert exercise_wildcard("aa", "*") == True
# assert exercise_wildcard("cb", "?a") == False
# assert exercise_wildcard("adceb", "*a*b") == True
# assert exercise_wildcard("acdcb", "a*c?b") == False

## 17. Gợi ý theo mức

**Bài 1**
- Mức 1: tách pattern thành token, hỏi "mỗi `x*` ăn bao nhiêu ký tự?".
- Mức 2: `".*"` khớp mọi thứ; `".."` cần **đúng 2** ký tự.
- Mức 3: câu 5: `.*` ≥ 0 ký tự + `..` đúng 2 ký tự + `a*` ≥ 0 → cần ít nhất 2 ký tự.

**Bài 2**
- Mức 1: viết hàng 0 trước (chỉ cột `*` có thể là `T`).
- Mức 2: cột `*` (j = 2): `dp[i][2] = dp[i][0] or ('a' khớp s[i-1] and dp[i-1][2])`.
- Mức 3: cột cuối `a` (j = 3): `dp[i][3] = (s[i-1] == 'a') and dp[i-1][2]`.

**Bài 3**
- Mức 1: chép bảng "slicing → chỉ số" ở mục 6.
- Mức 2: `if j == n: return i == m`.
- Mức 3: `return dp(i, j + 2) or (first_match and dp(i + 1, j))` cho nhánh `'*'`.

**Bài 4**
- Mức 1: nhớ ý nghĩa: `dp[i][j]` là `s[:i]` vs `p[:j]`, ký tự cuối là `s[i-1]`, `p[j-1]`.
- Mức 2: hàng 0 lặp `j` từ 2 tới `n`.
- Mức 3: cột `'*'`: `dp[i][j-2] or (p[j-2] in (s[i-1], ".") and dp[i-1][j])`.

**Bài 5**
- Mức 1: `x+` = "ăn ít nhất 1 ký tự bằng `x`".
- Mức 2: sau khi ăn 1 ký tự, hoặc **dừng** (bỏ token, `j + 2`), hoặc **ăn tiếp** (giữ token, `j`).
- Mức 3: `return first_match and (dp(i + 1, j + 2) or dp(i + 1, j))`.

**Bài 6**
- Mức 1: khung giống hệt Cách 3, chỉ đổi công thức.
- Mức 2: `'*'` khớp rỗng → `dp[i][j-1]`; `'*'` ăn thêm `s[i-1]` (vẫn giữ `'*'`) → `dp[i-1][j]`.
- Mức 3: hàng 0: `dp[0][j] = p[j-1] == '*' and dp[0][j-1]`.

## 18. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1 và Bài 2

Bài 1: `False`, `True`, `False`, `True`, `False`, `True`. Cell dưới kiểm chứng bằng `Solution` và `re.fullmatch`, rồi in bảng DP của Bài 2.

Bài 2: đáp án `dp[2][3] = T`.
- `dp[1][2]` (`"a"` vs `"a*"`): T nhờ nhánh **≥ 1 lần** (`'a'` khớp và `dp[0][2] = T`).
- `dp[2][2]` (`"aa"` vs `"a*"`): T nhờ nhánh **≥ 1 lần** (`dp[1][2] = T`).
- `dp[2][3]` (`"aa"` vs `"a*a"`): `'a'` khớp `'a'` và `dp[1][2] = T` → T. Đây chính là "để `a*` ăn 1 chữ, token `a` ăn chữ còn lại" — điều mà greedy bỏ lỡ.

In [30]:
quiz = [("aa", "a"), ("aa", ".*"), ("ab", ".*c"), ("aab", "c*a*b"), ("a", ".*..a*"), ("aaa", "ab*a*c*a")]
answers = [False, True, False, True, False, True]
for (s, p), ans in zip(quiz, answers):
    assert Solution().isMatch(s, p) == ans == (re.fullmatch(p, s) is not None)
    print(f"s={s!r:6} p={p!r:11} -> {ans}")

print("\nBài 2 — bảng DP cho s='aa', p='a*a':")
print_dp_table("aa", "a*a")

s='aa'   p='a'         -> False
s='aa'   p='.*'        -> True
s='ab'   p='.*c'       -> False
s='aab'  p='c*a*b'     -> True
s='a'    p='.*..a*'    -> False
s='aaa'  p='ab*a*c*a'  -> True

Bài 2 — bảng DP cho s='aa', p='a*a':
      ''   a   *   a
  ''   T   .   T   .
   a   .   T   T   T
   a   .   .   T   T


In [31]:
# ===== Đáp án Bài 3 và Bài 4 =====
def answer_is_match_memo(s: str, p: str) -> bool:
    m, n = len(s), len(p)

    @lru_cache(maxsize=None)
    def dp(i: int, j: int) -> bool:
        if j == n:
            return i == m
        first_match = i < m and p[j] in (s[i], ".")
        if j + 1 < n and p[j + 1] == "*":
            return dp(i, j + 2) or (first_match and dp(i + 1, j))
        return first_match and dp(i + 1, j + 1)

    return dp(0, 0)


def answer_is_match_dp(s: str, p: str) -> bool:
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for j in range(2, n + 1):
        if p[j - 1] == "*":
            dp[0][j] = dp[0][j - 2]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == "*":
                dp[i][j] = dp[i][j - 2] or (p[j - 2] in (s[i - 1], ".") and dp[i - 1][j])
            else:
                dp[i][j] = p[j - 1] in (s[i - 1], ".") and dp[i - 1][j - 1]
    return dp[m][n]


assert answer_is_match_memo("aa", "a") == False
assert answer_is_match_memo("aa", "a*") == True
assert answer_is_match_memo("aab", "c*a*b") == True
assert answer_is_match_memo("mississippi", "mis*is*p*.") == False
assert answer_is_match_dp("b", "a*b") == True
assert answer_is_match_dp("ab", ".*c") == False
assert answer_is_match_dp("aaa", "ab*a*c*a") == True
print("Bài 3, Bài 4: PASS")

Bài 3, Bài 4: PASS


### Đáp án Bài 5 — thêm `'+'`

Token `x+` bắt buộc ăn ít nhất 1 ký tự: `first_match` phải đúng, sau đó hoặc **dừng** (`j + 2`), hoặc **ăn tiếp** với chính `x+` (`j`). Ta kiểm tra chéo 3000 ca ngẫu nhiên với `re.fullmatch`.

In [32]:
def answer_is_match_plus(s: str, p: str) -> bool:
    m, n = len(s), len(p)

    @lru_cache(maxsize=None)
    def dp(i: int, j: int) -> bool:
        if j == n:
            return i == m
        first_match = i < m and p[j] in (s[i], ".")
        if j + 1 < n and p[j + 1] == "*":
            return dp(i, j + 2) or (first_match and dp(i + 1, j))
        if j + 1 < n and p[j + 1] == "+":
            return first_match and (dp(i + 1, j + 2) or dp(i + 1, j))
        return first_match and dp(i + 1, j + 1)

    return dp(0, 0)


assert answer_is_match_plus("aa", "a+") == True
assert answer_is_match_plus("b", "a+b") == False
assert answer_is_match_plus("ab", ".+") == True
assert answer_is_match_plus("aab", "a+a*b+") == True

random.seed(5)
for _ in range(3000):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 7)))
    p = "".join(random.choice("ab.") + random.choice(["", "*", "+"]) for _ in range(random.randint(1, 5)))
    assert answer_is_match_plus(s, p) == (re.fullmatch(p, s) is not None), (s, p)
print("Bài 5: PASS (kể cả 3000 test ngẫu nhiên)")

Bài 5: PASS (kể cả 3000 test ngẫu nhiên)


### Đáp án Bài 6 — Wildcard Matching

Khác biệt duy nhất so với Cách 3 nằm ở công thức cho `'*'`: vì `'*'` đứng một mình, "0 lần" là `dp[i][j-1]` (bỏ **1** ký tự pattern, không phải 2), và "ăn thêm" không cần kiểm tra ký tự nào.

In [33]:
import fnmatch

def answer_wildcard(s: str, p: str) -> bool:
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for j in range(1, n + 1):
        dp[0][j] = p[j - 1] == "*" and dp[0][j - 1]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == "*":
                dp[i][j] = dp[i][j - 1] or dp[i - 1][j]
            else:
                dp[i][j] = p[j - 1] in (s[i - 1], "?") and dp[i - 1][j - 1]
    return dp[m][n]


assert answer_wildcard("aa", "a") == False
assert answer_wildcard("aa", "*") == True
assert answer_wildcard("cb", "?a") == False
assert answer_wildcard("adceb", "*a*b") == True
assert answer_wildcard("acdcb", "a*c?b") == False

random.seed(44)
for _ in range(3000):
    s = "".join(random.choice("ab") for _ in range(random.randint(0, 7)))
    p = "".join(random.choice("ab?*") for _ in range(random.randint(0, 6)))
    assert answer_wildcard(s, p) == fnmatch.fnmatchcase(s, p), (s, p)
print("Bài 6: PASS (kể cả 3000 test ngẫu nhiên so với fnmatch)")

Bài 6: PASS (kể cả 3000 test ngẫu nhiên so với fnmatch)


## 19. Template ngắn nên nhớ

**Bản memo (viết nhanh nhất từ suy nghĩ đệ quy):**

```python
from functools import lru_cache

class Solution:
    def isMatch(self, s: str, p: str) -> bool:
        m, n = len(s), len(p)

        @lru_cache(maxsize=None)
        def dp(i, j):                      # s[i:] khớp p[j:]?
            if j == n:
                return i == m
            first_match = i < m and p[j] in (s[i], ".")
            if j + 1 < n and p[j + 1] == "*":
                return dp(i, j + 2) or (first_match and dp(i + 1, j))
            return first_match and dp(i + 1, j + 1)

        return dp(0, 0)
```

**Bản bảng bottom-up — 3 dòng cần thuộc:**

```text
dp[0][0] = True ;  dp[0][j] = p[j-1]=='*' and dp[0][j-2]
p[j-1] == '*' :  dp[i][j] = dp[i][j-2]  or  (p[j-2] khớp s[i-1] and dp[i-1][j])
ngược lại     :  dp[i][j] = p[j-1] khớp s[i-1]  and  dp[i-1][j-1]
```

Câu thần chú: **"`x*` = bỏ nó (0 lần) HOẶC ăn một ký tự và giữ nó lại."**

## 20. Checklist tự kiểm tra trước khi submit

- Mình có đọc `'*'` cùng ký tự **đứng trước** nó (token `x*`) chứ không phải một ký tự độc lập?
- Base case có dựa vào **pattern hết** (`j == n` / `not p`) chứ không phải `s` hết?
- Có bảo vệ `s[i]` / `s[0]` khi `s` đã hết (`i < m and ...` / `bool(s) and ...`)?
- Nhánh "≥ 1 lần" có **giữ nguyên** token `x*` (`dp(i + 1, j)` / `dp[i-1][j]`) không?
- Nhánh "0 lần" có nhảy **2** ký tự pattern (`j + 2` / `j - 2`) không?
- Với bảng DP: đã khởi tạo **hàng 0** cho các pattern kiểu `"a*b*"` chưa? Đã tạo bảng bằng list comprehension (không dùng `* (m + 1)`) chưa?
- Chỉ số tiền tố: ô `dp[i][j]` dùng `s[i-1]`, `p[j-1]` (lệch 1) — đã kiểm tra chưa?
- Đã test 3 ví dụ của đề + ca bẫy greedy (`"aa"`, `"a*a"`) + pattern có `'*'` đầu (`"c*a*b"`) chưa?